# Tweezer baseline

The *Baseline* section of `rearrangement-transport-test.ipynb` (`rearrangement-optimisation-RnD`) as a queued experiment, `pytweezer/experiments/tweezer_baseline.py:TweezerBaseline`: shots of `RbTweezerBasic2026_2` with the tweezer VVAs at 0 and a background drop, two ImagEM X2 images per shot, then the trap-site and loading analysis.

The SLM is not touched: upload the 13×13, 5 µm phasemask first.

In [ ]:
from pytweezer.analysis import analysis as an
from pytweezer.experiment import Dataset, Scan
from pytweezer.experiment.client import submit, wait
from pytweezer.experiment.experiment_parameter_manager import ExpParameterManager
from pytweezer.experiments.tweezer_baseline import TweezerBaseline
import numpy as np

ExpParams = ExpParameterManager()

## Baseline

In [ ]:
n_iterations = 100
rid = submit(TweezerBaseline, Scan(repetitions=n_iterations), label="Baseline 13x13 5um")
task = wait(rid)
assert task.status == "completed", f"task {rid} {task.status}: {task.error}"

In [ ]:
ds = Dataset(rid)
ds.summarise()
data = ds.get_dict()
imgs1, imgs2 = data["first_image"], data["second_image"]
img_average = an.tweezer_show_bg_subtracted(imgs1, imgs2, cmap='magma', show=True, vmaxfactor=0.6, show_grid=True)

# A simulated camera's conversion factor would overwrite the real one.
if not ds.attrs.get("simulated"):
    ExpParams.set_parameter("conversion_factor", ds.constants["conversion_factor"])
    ExpParams.set_parameter("conversion_offset", ds.constants["conversion_offset"])
    ExpParams.save_parameters()

Target sites of phasemask 1 (13×13, 5 µm, rotated 2.5°), as `PM.generate_weighted_array` lays them out.

In [ ]:
W1 = np.ones((13, 13))
spacing_um, angle_deg = 5.0, 2.5
x_half_span = (W1.shape[1] - 1) * spacing_um / 2
y_half_span = (W1.shape[0] - 1) * spacing_um / 2
x_grid, y_grid = np.meshgrid(
    np.linspace(-x_half_span, x_half_span, W1.shape[1]),
    np.linspace(-y_half_span, y_half_span, W1.shape[0]),
)
x_n1, y_n1 = an.rotate_coordinates(x_grid.flatten(), y_grid.flatten(), angle_deg)

In [ ]:
feature_size, window = 10, 5
grid_positions, detection_threshold = an.detect_trap_sites(img_average, 13*13, detection_step=100)
an.set_array_centre(grid_positions)
grid_positions = an.index_grid_positions(grid_positions, x_n1, y_n1)
an.visualize_results(img_average, grid_positions, margin=50, window_size=window, threshold=detection_threshold, vmaxfactor=0.5, bin_sharpness=20, bin_thresh_factor=0.9)
imgs_n_filtered = [an.morphological_tophat_high_pass(img, feature_size=feature_size) for img in imgs1]
photons, loading_probabilities, threshold_1, fidelity = an.get_array_loading_statistics(imgs_n_filtered, grid_positions, threshold=100, window_size=5, binning=60, 
                                                                                        show_histogram=True, threshold_detection=True, verbose=True, show_site_labels=True)

In [ ]:
# Analyse images (fliplr) with indexed grid positions - PSF
psf_window = 9

# Templates come from the raw frames - the matched filter needs no top-hat.
scorer = an.SiteScorer.from_images(imgs1, grid_positions, window_size=psf_window)
photons, loading_probabilities, threshold_1, fidelity = an.get_array_loading_statistics(
    imgs1, grid_positions, scorer=scorer, threshold_detection=True, binning=60)

scorer.threshold = threshold_1 * 1.0
print(f"PSF threshold {scorer.threshold:.2f} photons, detection fidelity {fidelity*100:.3f} %")